In [1]:
# Stacking Classifier
# Combine Decision Tree, Random Forest, and Logistic Regression
# using Logistic Regression as the meta-learner.

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
X, y = load_breast_cancer(return_X_y=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Base models
base_models = [
    ("decision_tree", DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    )),

    ("random_forest", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )),

    ("logistic_regression", make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    ))
]

# Meta-model
meta_model = LogisticRegression(max_iter=1000)

# Stacking classifier
stack_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    stack_method="predict_proba"
)

# Train
stack_model.fit(X_train, y_train)

# Predict
y_pred = stack_model.predict(X_test)

# Evaluate
print("Stacking Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Stacking Accuracy: 0.9824561403508771
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [2]:
# Blending Classifier
# Train base models on one split and a meta-model on holdout predictions.

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Load data
X, y = load_breast_cancer(return_X_y=True)

# First split: 80% development, 20% final test
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: base training and blending validation
X_base, X_blend, y_base, y_blend = train_test_split(
    X_dev, y_dev, test_size=0.25,
    random_state=42, stratify=y_dev
)

# Base models
models = [
    DecisionTreeClassifier(max_depth=5, random_state=42),
    RandomForestClassifier(n_estimators=100, random_state=42),
    make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    )
]

# Train base models on base training data
for model in models:
    model.fit(X_base, y_base)

# Generate blending predictions
blend_predictions = [
    model.predict_proba(X_blend)[:, 1]
    for model in models
]

# Create meta-training dataset
X_meta = list(zip(*blend_predictions))
X_meta = __import__("numpy").array(X_meta)

# Train meta-model
meta_model = LogisticRegression()
meta_model.fit(X_meta, y_blend)

# Predict on unseen test data
test_predictions = [
    model.predict_proba(X_test)[:, 1]
    for model in models
]

X_meta_test = __import__("numpy").array(
    list(zip(*test_predictions))
)

y_pred = meta_model.predict(X_meta_test)

print("Blending Accuracy:", accuracy_score(y_test, y_pred))

Blending Accuracy: 0.9473684210526315
